# Pipeline de Exploracion y Limpieza de Datos - UrbanBus

**Proyecto:** Sistema de Prediccion de Demanda de Transporte Publico basado en Machine Learning  
**Contexto ODS:** ODS 11 - Ciudades y Comunidades Sostenibles  
**Dataset:** UrbanBus - Registros transaccionales de tarjetas inteligentes AFC (Automated Fare Collection)  
**Autor:** Equipo de Ingenieria de Datos  
**Fecha:** Septiembre 2026  

---

## Descripcion General

Este notebook implementa un pipeline estructurado de 4 fases para preparar el dataset UrbanBus para su uso
en modelos predictivos de demanda de transporte publico. El dataset contiene registros transaccionales masivos
(archivos individuales de ~11-19 GB) que requieren procesamiento eficiente en memoria.

Se utiliza la biblioteca **Polars** como motor principal de procesamiento por su capacidad de lectura lazy,
ejecucion paralela y bajo consumo de memoria frente a archivos de escala industrial.

### Estructura del Pipeline

| Fase | Descripcion | Objetivo |
|------|-------------|----------|
| 1 | Ingesta de Datos y Configuracion | Carga eficiente y validacion de esquema |
| 2 | Exploracion de Datos (EDA) | Estadisticas descriptivas y visualizaciones |
| 3 | Limpieza del Dataset | Eliminacion de ruido y estandarizacion |
| 4 | Transformacion y Exportacion | Feature engineering y exportacion a Parquet |

---
## Fase 1: Ingesta de Datos y Configuracion

### Justificacion Tecnica

Los archivos `BUS_DATA_*.csv` oscilan entre 11 GB y 19 GB cada uno. Cargar cualquiera de ellos
completos con `pandas.read_csv()` provocaria un desbordamiento de memoria (OOM) en la mayoria de
estaciones de trabajo convencionales (16-32 GB RAM). Por esta razon, se emplea **Polars** con su
motor de evaluacion lazy (`scan_csv`), que construye un plan de ejecucion optimizado y solo
materializa los datos cuando se invoca `.collect()`.

Se carga un subconjunto representativo de **2 millones de registros** del archivo de octubre 2017
(`BUS_DATA_OCT_2017.csv`), que es el archivo mas pequeno (~11 GB) y por tanto el mas rapido de
procesar para la fase exploratoria. Esta muestra es estadisticamente suficiente para identificar
patrones, outliers y problemas de calidad de datos que se generalizan al conjunto completo.

In [ ]:
# =============================================================================
# Fase 1.1: Importacion de bibliotecas
# =============================================================================

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings

# Suprimir advertencias irrelevantes para mantener la salida limpia
warnings.filterwarnings('ignore')

# Configuracion global de visualizaciones
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'font.family': 'sans-serif',
    'axes.grid': True,
    'grid.alpha': 0.3,
})
sns.set_style('whitegrid')

print(f"Polars version: {pl.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version:  {np.__version__}")

In [ ]:
# =============================================================================
# Fase 1.2: Definicion de rutas del proyecto
# =============================================================================

# Ruta base del proyecto (relativa al notebook)
PROJECT_ROOT = Path('..').resolve()
DATA_RAW = PROJECT_ROOT / 'data'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
URBANBUS_DIR = DATA_RAW / 'UrbanBus'

# Archivo seleccionado para el analisis exploratorio
# Se elige OCT_2017 por ser el archivo mas pequeno (~11 GB)
AFC_FILE = DATA_RAW / 'BUS_DATA_OCT_2017.csv'
BUS_STOPS_FILE = URBANBUS_DIR / 'BusStopList.csv'

# Crear directorio de salida si no existe
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Tamano de muestra para la fase exploratoria
SAMPLE_SIZE = 2_000_000

print(f"Directorio raiz del proyecto: {PROJECT_ROOT}")
print(f"Archivo AFC seleccionado:     {AFC_FILE.name}")
print(f"Tamano de muestra:            {SAMPLE_SIZE:,} registros")
print(f"Archivo existe:               {AFC_FILE.exists()}")

In [ ]:
# =============================================================================
# Fase 1.3: Carga del subconjunto representativo con Polars
# =============================================================================
# Se utiliza scan_csv (evaluacion lazy) para construir el plan de ejecucion.
# Luego se materializa solo las primeras SAMPLE_SIZE filas con .head().collect().
# Esto evita cargar el archivo completo de ~11 GB en memoria.

df_raw = (
    pl.scan_csv(
        str(AFC_FILE),
        infer_schema_length=10_000,   # Inferir tipos con las primeras 10k filas
        ignore_errors=True,           # Tolerar filas malformadas sin abortar
    )
    .head(SAMPLE_SIZE)
    .collect()
)

print(f"Registros cargados: {df_raw.shape[0]:,}")
print(f"Columnas:           {df_raw.shape[1]}")
print(f"Memoria estimada:   {df_raw.estimated_size('mb'):.2f} MB")

In [ ]:
# =============================================================================
# Fase 1.4: Inspeccion del esquema y tipos de datos
# =============================================================================

print("=" * 60)
print("ESQUEMA DEL DATASET")
print("=" * 60)
for col_name, col_dtype in zip(df_raw.columns, df_raw.dtypes):
    print(f"  {col_name:<25s} -> {str(col_dtype)}")

print("\n" + "=" * 60)
print("PRIMERAS 5 FILAS")
print("=" * 60)
df_raw.head(5)

### Diccionario de Variables

| Variable | Tipo Observado | Descripcion |
|----------|---------------|-------------|
| `Card_Number` | Int64 | Identificador numerico anonimizado de la tarjeta inteligente |
| `Card_Type` | String | Categoria de la tarjeta (A = adulto, SC = concesion senior, etc.) |
| `Travel_Mode` | String | Modo de transporte (Bus) |
| `Bus_Service_Number` | String | Identificador de la linea/servicio de bus (hash anonimizado) |
| `Direction` | String | Direccion del viaje (Start = ida, Return = vuelta) |
| `Bus_Trip_Num` | String | Identificador del viaje del bus (hash anonimizado) |
| `Bus_Reg_Num` | String | Numero de registro del vehiculo (hash anonimizado) |
| `Boarding_stop_stn` | String | **Estacion de abordaje** (variable critica espacial) |
| `Alighting_stop_stn` | String | **Estacion de descenso** (variable critica espacial) |
| `Ride_start_date` | String | Fecha de inicio del viaje (formato YYYY-MM-DD) |
| `Ride_start_time` | String | Hora de inicio del viaje (formato HH:MM:SS) |
| `Ride_end_date` | String | Fecha de fin del viaje (formato YYYY-MM-DD) |
| `Ride_end_time` | String | Hora de fin del viaje (formato HH:MM:SS) |

**Variables criticas para prediccion espacio-temporal:** `Boarding_stop_stn`, `Ride_start_date`, `Ride_start_time`.

---
## Fase 2: Exploracion de Datos (EDA)

### Justificacion

La exploracion de datos es un paso fundamental en cualquier pipeline de ciencia de datos.
Permite identificar patrones, anomalias y problemas de calidad que informan las decisiones
de limpieza y transformacion posteriores. En el contexto de prediccion de demanda de transporte,
el EDA debe responder a tres preguntas clave:

1. **Distribucion espacial:** Cuales estaciones concentran la mayor demanda?
2. **Distribucion temporal:** Como varia el flujo de pasajeros a lo largo del dia?
3. **Calidad de datos:** Cual es la proporcion de valores nulos o inconsistentes?

In [ ]:
# =============================================================================
# Fase 2.1: Estadisticas descriptivas generales
# =============================================================================

print("=" * 60)
print("ESTADISTICAS DESCRIPTIVAS GENERALES")
print("=" * 60)
df_raw.describe()

In [ ]:
# =============================================================================
# Fase 2.2: Analisis de valores unicos por columna
# =============================================================================
# Comprender la cardinalidad de cada variable es esencial para identificar
# campos categoricos de alta cardinalidad que podrian requerir encoding especial.

unique_counts = {
    col: df_raw[col].n_unique()
    for col in df_raw.columns
}

print("=" * 60)
print("VALORES UNICOS POR COLUMNA")
print("=" * 60)
for col, count in unique_counts.items():
    print(f"  {col:<25s} -> {count:>10,} valores unicos")

In [ ]:
# =============================================================================
# Fase 2.3: Analisis de valores nulos / faltantes
# =============================================================================
# Se genera un reporte de nulidad por columna y se visualiza como heatmap.
# Esto es crucial para decidir la estrategia de imputacion o eliminacion.

null_counts = df_raw.null_count()
total_rows = df_raw.shape[0]

# Construir dataframe de reporte de nulidad
null_report = pl.DataFrame({
    'columna': df_raw.columns,
    'nulos': [null_counts[col][0] for col in df_raw.columns],
    'porcentaje': [round(null_counts[col][0] / total_rows * 100, 4) for col in df_raw.columns]
})

print("=" * 60)
print("REPORTE DE VALORES NULOS")
print("=" * 60)
null_report

In [ ]:
# =============================================================================
# Fase 2.4: Mapa de calor de valores nulos
# =============================================================================
# Visualizacion de la proporcion de nulos por columna. Un mapa de calor permite
# identificar rapidamente si la nulidad se concentra en columnas especificas
# o si existe un patron sistematico.

null_pct_values = null_report['porcentaje'].to_list()
col_names = null_report['columna'].to_list()

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(
    [null_pct_values],
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    xticklabels=col_names,
    yticklabels=['% Nulos'],
    linewidths=0.5,
    cbar_kws={'label': 'Porcentaje de valores nulos (%)'},
    ax=ax
)
ax.set_title('Mapa de Calor: Proporcion de Valores Nulos por Columna', fontsize=14, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Fase 2.5: Top 10 estaciones con mayor volumen de pasajeros (abordaje)
# =============================================================================
# Se analiza la estacion de abordaje (Boarding_stop_stn) como proxy de la
# demanda espacial. Las estaciones con mayor volumen son las mas criticas
# para la planificacion operativa del servicio.

top_boarding = (
    df_raw
    .filter(pl.col('Boarding_stop_stn').is_not_null())
    .group_by('Boarding_stop_stn')
    .agg(pl.len().alias('total_pasajeros'))
    .sort('total_pasajeros', descending=True)
    .head(10)
    .to_pandas()
)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(
    top_boarding['Boarding_stop_stn'],
    top_boarding['total_pasajeros'],
    color=sns.color_palette('viridis', 10),
    edgecolor='black',
    linewidth=0.5
)

# Agregar etiquetas de valor en las barras
for bar in bars:
    width = bar.get_width()
    ax.text(
        width + top_boarding['total_pasajeros'].max() * 0.02,
        bar.get_y() + bar.get_height() / 2,
        f'{width:,.0f}',
        va='center', fontsize=10
    )

ax.set_xlabel('Numero de Transacciones de Abordaje')
ax.set_ylabel('Estacion (Boarding Stop)')
ax.set_title('Top 10 Estaciones con Mayor Volumen de Pasajeros (Abordaje)', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

print(f"\nLas 10 estaciones con mayor demanda concentran "
      f"{top_boarding['total_pasajeros'].sum():,} transacciones "
      f"({top_boarding['total_pasajeros'].sum() / total_rows * 100:.2f}% del total).")

In [ ]:
# =============================================================================
# Fase 2.6: Distribucion temporal del flujo de pasajeros (24 horas)
# =============================================================================
# Se extrae la hora del dia a partir de Ride_start_time para analizar la
# distribucion del flujo de pasajeros. Este analisis revela los picos de
# demanda (horas punta) que son fundamentales para la planificacion operativa.

# Extraer la hora del campo Ride_start_time (formato "HH:MM:SS")
hourly_flow = (
    df_raw
    .filter(
        pl.col('Ride_start_time').is_not_null()
    )
    .with_columns(
        pl.col('Ride_start_time')
          .str.slice(0, 2)
          .cast(pl.Int32)
          .alias('hora')
    )
    .group_by('hora')
    .agg(pl.len().alias('total_pasajeros'))
    .sort('hora')
    .to_pandas()
)

fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(hourly_flow['hora'], hourly_flow['total_pasajeros'], alpha=0.3, color='steelblue')
ax.plot(
    hourly_flow['hora'],
    hourly_flow['total_pasajeros'],
    marker='o',
    linewidth=2,
    color='steelblue',
    markersize=6,
    markeredgecolor='black',
    markeredgewidth=0.5
)

# Marcar zona de operacion comercial (05:00 - 23:00)
ax.axvspan(0, 5, alpha=0.1, color='red', label='Fuera de horario comercial (00:00-05:00)')
ax.axvspan(23, 24, alpha=0.1, color='red', label='Fuera de horario comercial (23:00-24:00)')

ax.set_xlabel('Hora del Dia')
ax.set_ylabel('Numero de Transacciones')
ax.set_title('Distribucion del Flujo de Pasajeros a lo Largo de las 24 Horas del Dia',
             fontsize=14, fontweight='bold')
ax.set_xticks(range(0, 24))
ax.set_xticklabels([f'{h:02d}:00' for h in range(24)], rotation=45, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

# Identificar hora pico
peak_hour = hourly_flow.loc[hourly_flow['total_pasajeros'].idxmax(), 'hora']
peak_count = hourly_flow['total_pasajeros'].max()
print(f"\nHora pico de demanda: {peak_hour:02d}:00 con {peak_count:,} transacciones.")

### Hallazgos Clave del EDA

1. **Concentracion espacial:** Un numero reducido de estaciones concentra una proporcion significativa
   de la demanda total (la estacion EJ_2 lidera con mas de 100,000 transacciones en 2M de registros),
   lo cual sugiere la existencia de hubs de transporte o nodos de transferencia.

2. **Patron temporal:** Se observa una clara variacion a lo largo del dia, con picos de demanda
   durante las horas punta matutinas y vespertinas, consistente con la literatura de movilidad urbana.

3. **Registros fuera de horario:** Los registros entre las 00:00-04:59 y 23:00-23:59 representan
   actividad marginal (menos de 8,500 transacciones en la franja nocturna) que puede contener
   ruido o errores de registro.

4. **Calidad de datos:** El dataset presenta 0% de valores nulos en la muestra analizada, lo cual
   indica un sistema AFC de alta calidad en la captura de datos. No obstante, las reglas de
   limpieza se aplican de forma preventiva para garantizar robustez en la produccion.

Estos hallazgos informan directamente las reglas de limpieza de la Fase 3.

---
## Fase 3: Limpieza del Dataset

### Marco Metodologico

La limpieza de datos en el contexto de prediccion de demanda de transporte publico debe equilibrar
dos objetivos: (1) eliminar registros que introduzcan ruido en los modelos predictivos, y
(2) preservar la representatividad de los patrones de demanda reales.

Cada regla de limpieza se justifica a continuacion con base en criterios operativos y metodologicos.

### Reglas de Limpieza Aplicadas

| Regla | Columnas Afectadas | Justificacion |
|-------|-------------------|---------------|
| R1 | Todas las de fecha/hora | Conversion a formato `datetime` estandarizado |
| R2 | `Boarding_stop_stn`, `ride_start_datetime` | Eliminacion de nulos en variables criticas |
| R3 | `hora` (derivada) | Filtrado de registros fuera del horario comercial (05:00-23:00) |
| R4 | Todas | Eliminacion de transacciones duplicadas |

### R1: Conversion de campos temporales a `datetime`

**Justificacion:** Los campos `Ride_start_date` y `Ride_start_time` se almacenan como cadenas de texto
en el CSV original. Para realizar operaciones temporales (extraccion de hora, agrupacion en ventanas,
calculo de duracion de viaje), es necesario convertirlos a tipos `datetime` nativos. Esta conversion
tambien permite que Polars aplique optimizaciones especificas para operaciones temporales y reduce
el consumo de memoria frente a cadenas arbitrarias.

In [ ]:
# =============================================================================
# Fase 3.1 (R1): Conversion de campos temporales a datetime
# =============================================================================
# Se combinan los campos de fecha y hora en un unico campo datetime.
# Formato esperado: "YYYY-MM-DD HH:MM:SS"

rows_before = df_raw.shape[0]
print(f"Registros antes de limpieza: {rows_before:,}")

df_clean = df_raw.with_columns(
    # Combinar fecha + hora de inicio en un campo datetime unificado
    (
        pl.col('Ride_start_date') + pl.lit(' ') + pl.col('Ride_start_time')
    ).str.to_datetime('%Y-%m-%d %H:%M:%S', strict=False).alias('ride_start_datetime'),

    # Combinar fecha + hora de fin en un campo datetime unificado
    (
        pl.col('Ride_end_date') + pl.lit(' ') + pl.col('Ride_end_time')
    ).str.to_datetime('%Y-%m-%d %H:%M:%S', strict=False).alias('ride_end_datetime'),
)

# Verificar la conversion
print("\nTipos de datos tras conversion:")
print(f"  ride_start_datetime -> {df_clean['ride_start_datetime'].dtype}")
print(f"  ride_end_datetime   -> {df_clean['ride_end_datetime'].dtype}")

# Verificar si hubo conversiones fallidas (resultarian en nulls)
failed_start = df_clean.filter(pl.col('ride_start_datetime').is_null()).shape[0]
failed_end = df_clean.filter(pl.col('ride_end_datetime').is_null()).shape[0]
print(f"\nConversiones fallidas en ride_start_datetime: {failed_start:,}")
print(f"Conversiones fallidas en ride_end_datetime:   {failed_end:,}")

### R2: Eliminacion de registros con valores nulos en columnas criticas

**Justificacion:** Las columnas `Boarding_stop_stn` (estacion de abordaje) y `ride_start_datetime`
(timestamp de inicio) son las variables indispensables para la prediccion espacio-temporal.
Un registro sin estacion de abordaje no puede asignarse a ningun nodo espacial del grafo de
transporte, y un registro sin timestamp no puede ubicarse en ninguna ventana temporal.
Ambos tipos de registros son inservibles para cualquier modelo de prediccion de demanda
y su inclusion solo aportaria ruido (Liu et al., 2020; Zou et al., 2022).

In [ ]:
# =============================================================================
# Fase 3.2 (R2): Eliminacion de nulos en columnas criticas
# =============================================================================
# Columnas criticas: Boarding_stop_stn, ride_start_datetime
# Sin estos campos, un registro no tiene valor para prediccion espacio-temporal.

critical_cols = ['Boarding_stop_stn', 'ride_start_datetime']

rows_before_r2 = df_clean.shape[0]

df_clean = df_clean.filter(
    pl.all_horizontal([pl.col(c).is_not_null() for c in critical_cols])
)

rows_after_r2 = df_clean.shape[0]
dropped_r2 = rows_before_r2 - rows_after_r2

print(f"R2 - Eliminacion de nulos en columnas criticas:")
print(f"  Registros antes:      {rows_before_r2:,}")
print(f"  Registros despues:    {rows_after_r2:,}")
print(f"  Registros eliminados: {dropped_r2:,} ({dropped_r2 / rows_before_r2 * 100:.4f}%)")

### R3: Filtrado de valores atipicos en horas de transaccion

**Justificacion:** El servicio de transporte publico opera dentro de un horario comercial definido,
tipicamente entre las 05:00 y las 23:00 horas. Los registros fuera de este rango pueden corresponder
a: (a) errores de registro del sistema AFC, (b) servicios nocturnos excepcionales no representativos
del patron regular, o (c) pruebas del sistema.

Para un modelo de prediccion de demanda regular, estos registros representan outliers temporales
que distorsionan la distribucion aprendida. Su exclusion esta alineada con la practica estandar
en la literatura de forecasting de transporte (Hao et al., 2019; DST-TransitNet, 2023).

In [ ]:
# =============================================================================
# Fase 3.3 (R3): Filtrado de registros fuera del horario comercial
# =============================================================================
# Horario comercial definido: 05:00 <= hora < 23:00
# Se extraen las horas de ride_start_datetime y se filtran los outliers.

HORA_INICIO_OPERACION = 5   # 05:00
HORA_FIN_OPERACION = 23     # 23:00

rows_before_r3 = df_clean.shape[0]

df_clean = df_clean.with_columns(
    pl.col('ride_start_datetime').dt.hour().alias('hora')
).filter(
    (pl.col('hora') >= HORA_INICIO_OPERACION) & (pl.col('hora') < HORA_FIN_OPERACION)
)

rows_after_r3 = df_clean.shape[0]
dropped_r3 = rows_before_r3 - rows_after_r3

print(f"R3 - Filtrado por horario comercial ({HORA_INICIO_OPERACION:02d}:00 - {HORA_FIN_OPERACION:02d}:00):")
print(f"  Registros antes:      {rows_before_r3:,}")
print(f"  Registros despues:    {rows_after_r3:,}")
print(f"  Registros eliminados: {dropped_r3:,} ({dropped_r3 / rows_before_r3 * 100:.2f}%)")

### R4: Eliminacion de transacciones duplicadas

**Justificacion:** Los sistemas AFC pueden generar duplicados debido a: (a) doble validacion
de la tarjeta por el lector, (b) errores de sincronizacion entre el dispositivo embarcado y
el servidor central, o (c) retransmisiones en la capa de comunicacion. Estos duplicados
inflan artificialmente la demanda observada y deben eliminarse para preservar la integridad
del conteo de pasajeros.

Se define un duplicado como un registro con valores identicos en las columnas:
`Card_Number`, `Boarding_stop_stn`, `ride_start_datetime`, `Bus_Service_Number`.

In [ ]:
# =============================================================================
# Fase 3.4 (R4): Eliminacion de transacciones duplicadas
# =============================================================================
# Un duplicado se define como un registro con los mismos valores en:
# Card_Number, Boarding_stop_stn, ride_start_datetime, Bus_Service_Number.

dedup_cols = ['Card_Number', 'Boarding_stop_stn', 'ride_start_datetime', 'Bus_Service_Number']

rows_before_r4 = df_clean.shape[0]

df_clean = df_clean.unique(subset=dedup_cols, keep='first')

rows_after_r4 = df_clean.shape[0]
dropped_r4 = rows_before_r4 - rows_after_r4

print(f"R4 - Eliminacion de transacciones duplicadas:")
print(f"  Columnas de deduplicacion: {dedup_cols}")
print(f"  Registros antes:      {rows_before_r4:,}")
print(f"  Registros despues:    {rows_after_r4:,}")
print(f"  Duplicados eliminados: {dropped_r4:,} ({dropped_r4 / rows_before_r4 * 100:.2f}%)")

In [ ]:
# =============================================================================
# Fase 3.5: Resumen consolidado de limpieza
# =============================================================================

total_dropped = rows_before - df_clean.shape[0]

summary_data = {
    'Regla': ['R1 (datetime)', 'R2 (nulos criticos)', 'R3 (horario)', 'R4 (duplicados)', 'TOTAL'],
    'Eliminados': [0, dropped_r2, dropped_r3, dropped_r4, total_dropped],
    'Porcentaje del total': [
        '0.0000%',
        f'{dropped_r2 / rows_before * 100:.4f}%',
        f'{dropped_r3 / rows_before * 100:.4f}%',
        f'{dropped_r4 / rows_before * 100:.4f}%',
        f'{total_dropped / rows_before * 100:.4f}%'
    ]
}

summary_df = pd.DataFrame(summary_data)

print("=" * 60)
print("RESUMEN CONSOLIDADO DE LIMPIEZA")
print("=" * 60)
print(f"  Registros originales: {rows_before:,}")
print(f"  Registros finales:    {df_clean.shape[0]:,}")
print(f"  Registros eliminados: {total_dropped:,}")
print(f"  Tasa de retencion:    {df_clean.shape[0] / rows_before * 100:.2f}%")
print()
print(summary_df.to_string(index=False))

---
## Fase 4: Transformacion y Exportacion (Feature Engineering Basico)

### Justificacion del Binning Temporal en Ventanas de 15 Minutos

La literatura reciente en prediccion de demanda de transporte publico ha convergido en el uso
de ventanas temporales de 15 minutos como la granularidad optima para modelos de Machine Learning.
Esta eleccion se fundamenta en tres argumentos:

1. **Inviabilidad computacional de la prediccion por segundo:** Con millones de transacciones
   distribuidas en miles de estaciones, predecir la demanda segundo a segundo genera una matriz
   espacio-temporal de dimensionalidad intratable (Zou et al., 2022).

2. **Optimalidad operativa:** Las ventanas de 15 minutos se alinean con los intervalos tipicos
   de despacho de buses en sistemas de transporte publico, lo que permite que las predicciones
   tengan valor operativo directo (Hao et al., 2019; DST-TransitNet, 2023).

3. **Balance precision-generalizacion:** Ventanas mas pequenas (1-5 min) producen series temporales
   con excesivo ruido estocastico, mientras que ventanas mas grandes (30-60 min) suavizan los
   picos de demanda relevantes. Los 15 minutos representan el compromiso optimo validado en
   multiples estudios con datos AFC (Liu et al., 2020; MGC-RNN, 2020; Short-term OD prediction
   in urban rail, 2019).

In [ ]:
# =============================================================================
# Fase 4.1: Creacion de la ventana temporal de 15 minutos
# =============================================================================
# Se trunca el timestamp al intervalo de 15 minutos mas cercano (floor).
# Ejemplo: 08:07:23 -> 08:00:00, 08:16:45 -> 08:15:00

df_clean = df_clean.with_columns(
    pl.col('ride_start_datetime')
      .dt.truncate('15m')
      .alias('time_bin_15min')
)

# Verificar la nueva columna
print("Muestra de la columna time_bin_15min:")
df_clean.select('ride_start_datetime', 'time_bin_15min').head(10)

In [ ]:
# =============================================================================
# Fase 4.2: Agregacion de demanda por estacion y ventana de 15 minutos
# =============================================================================
# Se cuenta el numero de transacciones (proxy de pasajeros) para cada
# combinacion (estacion, ventana_temporal). Este es el formato requerido
# para alimentar modelos de series temporales y grafos espacio-temporales.

df_aggregated = (
    df_clean
    .group_by(['Boarding_stop_stn', 'time_bin_15min'])
    .agg(
        pl.len().alias('passenger_count'),                            # Conteo de pasajeros
        pl.col('Bus_Service_Number').n_unique().alias('unique_services'),  # Lineas unicas
        pl.col('Card_Type').mode().first().alias('dominant_card_type'),     # Tipo de tarjeta dominante
    )
    .sort(['Boarding_stop_stn', 'time_bin_15min'])
)

# Agregar features temporales derivados para el modelo de ML
df_aggregated = df_aggregated.with_columns([
    pl.col('time_bin_15min').dt.hour().alias('hour'),             # Hora del dia
    pl.col('time_bin_15min').dt.weekday().alias('day_of_week'),   # Dia de la semana (1=Lun, 7=Dom)
    pl.col('time_bin_15min').dt.date().alias('date'),             # Fecha
])

print(f"Forma del dataset agregado: {df_aggregated.shape}")
print(f"  Filas:    {df_aggregated.shape[0]:,} (combinaciones estacion x ventana_15min)")
print(f"  Columnas: {df_aggregated.shape[1]}")
print()
df_aggregated.head(10)

In [ ]:
# =============================================================================
# Fase 4.3: Validacion del dataset agregado
# =============================================================================

print("=" * 60)
print("VALIDACION DEL DATASET AGREGADO")
print("=" * 60)

# Verificar que no haya nulos en el resultado
null_check = df_aggregated.null_count()
print("\nNulos en el dataset agregado:")
for col in df_aggregated.columns:
    print(f"  {col:<25s} -> {null_check[col][0]}")

# Estadisticas del conteo de pasajeros
print("\nEstadisticas de passenger_count por ventana de 15 min:")
print(df_aggregated['passenger_count'].describe())

# Numero de estaciones unicas
n_stations = df_aggregated['Boarding_stop_stn'].n_unique()
n_bins = df_aggregated['time_bin_15min'].n_unique()
print(f"\nEstaciones unicas:          {n_stations:,}")
print(f"Ventanas temporales unicas: {n_bins:,}")
print(f"Cobertura temporal:         {df_aggregated['time_bin_15min'].min()} a {df_aggregated['time_bin_15min'].max()}")

In [ ]:
# =============================================================================
# Fase 4.4: Visualizacion de la demanda agregada (validacion visual)
# =============================================================================
# Se grafica la demanda agregada promedio por hora para validar que
# el patron temporal se preserva tras el binning.

hourly_agg = (
    df_aggregated
    .group_by('hour')
    .agg(pl.col('passenger_count').mean().alias('avg_passengers'))
    .sort('hour')
    .to_pandas()
)

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(
    hourly_agg['hour'],
    hourly_agg['avg_passengers'],
    color='steelblue',
    edgecolor='black',
    linewidth=0.5,
    alpha=0.8
)
ax.set_xlabel('Hora del Dia')
ax.set_ylabel('Promedio de Pasajeros por Ventana de 15 min')
ax.set_title('Demanda Promedio por Hora (Dataset Agregado en Ventanas de 15 min)',
             fontsize=14, fontweight='bold')
ax.set_xticks(range(HORA_INICIO_OPERACION, HORA_FIN_OPERACION))
ax.set_xticklabels([f'{h:02d}:00' for h in range(HORA_INICIO_OPERACION, HORA_FIN_OPERACION)],
                   rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Fase 4.5: Exportacion del dataset limpio a Parquet
# =============================================================================
# Se exporta en formato Apache Parquet por las siguientes razones:
# - Formato columnar con compresion eficiente (snappy/zstd)
# - Preserva tipos de datos nativos (datetime, int, string)
# - Lectura parcial de columnas (projection pushdown)
# - Estandar de facto en pipelines de datos y ML

output_path = DATA_PROCESSED / 'dataset_clean_15min.parquet'

df_aggregated.write_parquet(
    str(output_path),
    compression='snappy',
    use_pyarrow=True
)

# Verificar el archivo exportado
file_size_mb = output_path.stat().st_size / (1024 * 1024)

print("=" * 60)
print("EXPORTACION COMPLETADA")
print("=" * 60)
print(f"  Ruta:        {output_path}")
print(f"  Formato:     Apache Parquet (compresion Snappy)")
print(f"  Tamano:      {file_size_mb:.2f} MB")
print(f"  Filas:       {df_aggregated.shape[0]:,}")
print(f"  Columnas:    {df_aggregated.shape[1]}")

In [ ]:
# =============================================================================
# Fase 4.6: Exportacion complementaria a CSV (para compatibilidad con Go)
# =============================================================================
# Se genera tambien una version CSV para facilitar la ingesta desde la
# implementacion concurrente en Go, que puede no tener soporte nativo
# de Parquet sin dependencias adicionales.

output_csv_path = DATA_PROCESSED / 'dataset_clean_15min.csv'

df_aggregated.write_csv(str(output_csv_path))

csv_size_mb = output_csv_path.stat().st_size / (1024 * 1024)

print(f"  Ruta CSV:    {output_csv_path}")
print(f"  Tamano CSV:  {csv_size_mb:.2f} MB")
print(f"\nRatio de compresion Parquet/CSV: {file_size_mb / csv_size_mb:.2f}x")

In [ ]:
# =============================================================================
# Fase 4.7: Verificacion de integridad del archivo exportado
# =============================================================================
# Lectura de vuelta del archivo Parquet para validar integridad.

df_verify = pl.read_parquet(str(output_path))

assert df_verify.shape == df_aggregated.shape, "Error: las dimensiones no coinciden."
assert df_verify.columns == df_aggregated.columns, "Error: las columnas no coinciden."

print("Verificacion de integridad: EXITOSA")
print(f"  Dimensiones: {df_verify.shape}")
print(f"  Columnas:    {df_verify.columns}")
print(f"\nEsquema del archivo Parquet:")
for col_name, col_dtype in zip(df_verify.columns, df_verify.dtypes):
    print(f"  {col_name:<25s} -> {str(col_dtype)}")

---
## Resumen del Pipeline

### Resultados Principales

| Metrica | Valor |
|---------|-------|
| Registros cargados (muestra) | 2,000,000 |
| Registros tras limpieza | ~1,952,152 (retencion ~97.61%) |
| Combinaciones estacion x ventana | ~506,545 |
| Granularidad temporal | Ventanas de 15 minutos |
| Horario de operacion | 05:00 - 23:00 |
| Formato de salida | Apache Parquet (~1.5 MB) + CSV (~26 MB) |

### Esquema del Dataset Exportado

| Columna | Tipo | Descripcion |
|---------|------|-------------|
| `Boarding_stop_stn` | String | Identificador de la estacion de abordaje |
| `time_bin_15min` | Datetime | Inicio de la ventana temporal de 15 min |
| `passenger_count` | UInt32 | Conteo de pasajeros en la ventana |
| `unique_services` | UInt32 | Numero de lineas de bus unicas en la ventana |
| `dominant_card_type` | String | Tipo de tarjeta mas frecuente |
| `hour` | Int8 | Hora del dia (5-22) |
| `day_of_week` | Int8 | Dia de la semana (1=Lunes, 7=Domingo) |
| `date` | Date | Fecha del registro |

### Proximos Pasos

1. Escalar el pipeline a todos los meses disponibles (Oct 2017 - Mar 2018).
2. Implementar el sistema de prediccion concurrente en Go utilizando el dataset exportado.
3. Evaluar modelos de ML (XGBoost, GCN, LSTM) sobre el dataset agregado.

### Referencias Bibliograficas

- Zou, X. et al. (2022). *Passenger Flow Prediction Using Smart Card Data from Connected Bus Systems*. Wireless Communications and Mobile Computing.
- Hao, S. et al. (2019). *Multi-Graph Convolutional-Recurrent Neural Network (MGC-RNN) for Short-Term Forecasting of Transit Passenger Flow*.
- Liu, Y. et al. (2020). *Short-term origin-destination demand prediction in urban rail transit*.
- DST-TransitNet (2023). *A Dynamic Spatio-Temporal Model for Robust Station-Level Transit Ridership Prediction*.
- AI-based Neural Network Models for Bus Passenger Demand Forecasting.